# Notebook 01 — TrOCR Handwriting Recognition

This notebook explores and benchmarks **TrOCR** (`microsoft/trocr-base-handwritten`) as the
Phase 2 OCR engine, replacing the EasyOCR baseline from Phase 1.

## Architecture Overview
TrOCR is a **Vision Encoder–Decoder** model:
- **Encoder**: Vision Transformer (ViT) — images are split into non-overlapping 16×16 patches,
  linearly projected into patch embeddings, then processed by Transformer encoder layers.
- **Decoder**: GPT-2–style autoregressive Transformer — generates text tokens one by one
  using cross-attention to the visual feature map produced by the encoder.

Pre-trained on IAM handwriting dataset + SROIE + printed text data from MS COCO.

## What this notebook covers
1. Loading TrOCR and running inference on sample answer-sheet crops
2. Comparing TrOCR vs EasyOCR (Phase 1 baseline) on the same images
3. Visualising answer block detection (OpenCV projection)
4. Ablation: beam size effect on accuracy

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))

import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print(f'PyTorch: {torch.__version__}')
print(f'Device : {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 1. Load TrOCR

In [ ]:
CHECKPOINT = 'microsoft/trocr-base-handwritten'

print(f'Loading TrOCR from {CHECKPOINT} ...')
processor = TrOCRProcessor.from_pretrained(CHECKPOINT)
model     = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print('\nModel config summary:')
print(f'  Encoder: {model.config.encoder._name_or_path}')
print(f'  Decoder: {model.config.decoder._name_or_path}')
enc_params = sum(p.numel() for p in model.encoder.parameters())
dec_params = sum(p.numel() for p in model.decoder.parameters())
print(f'  Encoder params: {enc_params/1e6:.1f}M')
print(f'  Decoder params: {dec_params/1e6:.1f}M')
print(f'  Total params  : {(enc_params+dec_params)/1e6:.1f}M')

## 2. OCR Helper Function

In [ ]:
def trocr_predict(pil_image, num_beams=4, max_new_tokens=128):
    """
    Run TrOCR inference on a PIL image.
    
    The processor tokenizes the image into ViT patch tokens.
    generate() runs beam search: at each step it maintains `num_beams`
    candidate sequences and expands the one with highest cumulative log-prob.
    """
    pixel_values = processor(images=pil_image, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = model.generate(
            pixel_values,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

# Quick sanity check with a programmatically generated text image
from PIL import ImageDraw, ImageFont
test_img = Image.new('RGB', (400, 60), 'white')
draw     = ImageDraw.Draw(test_img)
draw.text((10, 10), 'Photosynthesis converts light to energy', fill='black')

result = trocr_predict(test_img)
print(f'Prediction: "{result}"')

## 3. Block Detection (OpenCV Horizontal Projection)

In [ ]:
from sheet_analyzer import locate_answer_blocks, draw_annotations, _binarize

# Replace with your own test image path
IMAGE_PATH = os.path.join('..', 'Automated_Edtech_Grading', 'sheet.png')

if os.path.exists(IMAGE_PATH):
    img    = cv2.imread(IMAGE_PATH)
    binary = _binarize(img)
    blocks = locate_answer_blocks(img)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Image')
    axes[1].imshow(binary, cmap='gray')
    axes[1].set_title('Binarized (Adaptive Threshold)')

    ann = draw_annotations(img, blocks)
    axes[2].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f'Detected Blocks ({len(blocks)})')

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Detected {len(blocks)} answer blocks.')
else:
    print(f'Test image not found at {IMAGE_PATH} — replace with your own path.')

## 4. Beam Search Ablation

In [ ]:
import time

test_img2 = Image.new('RGB', (500, 80), 'white')
draw2     = ImageDraw.Draw(test_img2)
draw2.text((10, 20), 'The mitochondria is the powerhouse of the cell', fill='black')

beam_results = []
for beams in [1, 2, 4, 8]:
    t0  = time.time()
    out = trocr_predict(test_img2, num_beams=beams)
    dt  = time.time() - t0
    beam_results.append({'beams': beams, 'time_s': round(dt, 2), 'output': out})
    print(f'  beams={beams:2d}  {dt:.2f}s  → "{out}"')

# Plot beam vs latency
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.bar([str(r['beams']) for r in beam_results], [r['time_s'] for r in beam_results], color='steelblue')
plt.xlabel('Num Beams')
plt.ylabel('Inference time (s)')
plt.title('TrOCR: Beam Size vs Inference Latency')
plt.tight_layout()
plt.show()

## 5. TrOCR vs EasyOCR Comparison

Run both engines on the same crops and compare character-level accuracy.

In [ ]:
# Requires: pip install easyocr
try:
    import easyocr
    easy_reader = easyocr.Reader(['en'], gpu=False)
    EASYOCR_AVAILABLE = True
    print('EasyOCR loaded (Phase 1 baseline)')
except ImportError:
    EASYOCR_AVAILABLE = False
    print('EasyOCR not installed — skipping comparison')

GROUND_TRUTH_SAMPLES = [
    'Photosynthesis is the process by which plants convert sunlight into glucose.',
    'Newton second law states force equals mass times acceleration.',
    'The water cycle consists of evaporation condensation and precipitation.',
]

def char_error_rate(pred, gt):
    """Simple CER: edit distance / len(gt)"""
    import difflib
    sm = difflib.SequenceMatcher(None, pred.lower(), gt.lower())
    return 1 - sm.ratio()

print('\n{:40s} {:>8s} {:>8s}'.format('Ground Truth (truncated)', 'TrOCR CER', 'Easy CER'))
print('-' * 60)
for gt in GROUND_TRUTH_SAMPLES:
    img_test = Image.new('RGB', (600, 70), 'white')
    draw_t   = ImageDraw.Draw(img_test)
    draw_t.text((10, 15), gt, fill='black')

    trocr_out = trocr_predict(img_test)
    cer_trocr = char_error_rate(trocr_out, gt)

    if EASYOCR_AVAILABLE:
        easy_results = easy_reader.readtext(np.array(img_test), detail=0, paragraph=True)
        easy_out = ' '.join(easy_results)
        cer_easy = char_error_rate(easy_out, gt)
    else:
        cer_easy = float('nan')

    print(f'{gt[:40]:40s} {cer_trocr:8.3f} {cer_easy:8.3f}')

## Summary

| Aspect | Phase 1 (EasyOCR) | Phase 2 (TrOCR) |
|--------|-------------------|------------------|
| Architecture | CNN backbone + CTC | ViT encoder + GPT-2 decoder |
| Decoding | CTC (no beam search) | Beam search (4 beams) |
| Handwriting | Good | Better (pre-trained on IAM) |
| Speed | Faster | Slower (~2–5× on CPU) |
| Model size | ~250 MB | ~400 MB |

Trade-off: TrOCR is more accurate for cursive/ambiguous handwriting at the cost of
higher latency. For Phase 2 (accuracy-focused evaluation), this trade-off is acceptable.